In [0]:
# Notebook parameters

params = {
    "vol_path": "",
    "input_dir": "",
    "output_dir": ""
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, "", "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:
import os
import time
from datetime import datetime
from tqdm.notebook import tqdm
import pydicom
import asyncio
from functools import lru_cache
from pyspark.sql import types as T
from pyspark.sql import functions as F
from glob import glob

In [0]:
# Databricks notebook
# pip install pydicom

import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pydicom.tag import Tag

# ------------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------------

if params["vol_path"] is not None and len(str(params["vol_path"])) > 0:
    ROOT_DIR = params["vol_path"]
else:
    ROOT_DIR = "/Volumes/1_inland/sectra/vone"

INPUT_ROOT = os.path.join(ROOT_DIR, params["input_dir"])
OUTPUT_ROOT = os.path.join(ROOT_DIR, params["output_dir"])


# ------------------------------------------------------------------------------
# Input file list (paths only)
# ------------------------------------------------------------------------------

files_df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .load(INPUT_ROOT)
    .select(["path", "length"])
)

# ------------------------------------------------------------------------------
# Output schema
# ------------------------------------------------------------------------------

schema = StructType([
    StructField("path", StringType(), False),
    StructField("output_path", StringType(), True),
    StructField("success", BooleanType(), False),
    StructField("error", StringType(), True),
])

# ------------------------------------------------------------------------------
# Tag anonymization logic
# ------------------------------------------------------------------------------

def should_anonymize(tag):
    """
    Covers:
    - (0010,10xx)
    - (0010,21xx)
    - Study/Series Date/Time (0008 group)
    """

    # Patient-related patterns
    if tag.group == 0x0010 and (
        0x1000 <= tag.element <= 0x10FF or
        0x2100 <= tag.element <= 0x21FF
    ):
        return True

    # Study / Series metadata
    if tag in {
        Tag(0x0008, 0x0020),  # StudyDate
        Tag(0x0008, 0x0030),  # StudyTime
        Tag(0x0008, 0x0021),  # SeriesDate
        Tag(0x0008, 0x0031),  # SeriesTime
        Tag(0x0008, 0x0023),  # ContentDate
        Tag(0x0008, 0x0033),  # ContentTime
        Tag(0x0008, 0x0050),  # AccessionNumber
    }:
        return True

    return False


def anonymize_value(vr):
    """
    Return DICOM-valid placeholder based on VR type.
    """

    if vr == "DA":
        return "19000101"   # valid DICOM date
    if vr == "TM":
        return "000000"     # valid DICOM time

    # Most text-like VRs
    return ""


# ------------------------------------------------------------------------------
# mapInPandas worker
# ------------------------------------------------------------------------------

def process_partition(iterator):
    import os
    import pandas as pd
    import pydicom

    for pdf in iterator:
        results = []

        for src in pdf["path"]:
            src = src[5:]
            try:
                rel = os.path.relpath(src, INPUT_ROOT)
                dst = os.path.join(OUTPUT_ROOT, rel)

                os.makedirs(os.path.dirname(dst), exist_ok=True)

                # Single read only
                ds = pydicom.dcmread(
                    src,
                    defer_size="1 KB",
                    force=True,
                )

                # Anonymize selected tags
                for elem in ds.iterall():
                    if should_anonymize(elem.tag):
                        elem.value = anonymize_value(elem.VR)

                # Write output
                ds.save_as(dst, write_like_original=True)

                results.append((src, dst, True, None))

            except Exception as e:
                results.append((src, None, False, str(e)))

        yield pd.DataFrame(
            results,
            columns=["path", "output_path", "success", "error"],
        )


# ------------------------------------------------------------------------------
# Execute
# ------------------------------------------------------------------------------

spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "500")

result_df = (
    files_df
    #.repartition(500)
    .mapInPandas(process_partition, schema=schema)
)

result_df.write.mode("overwrite").parquet(
    os.path.join(OUTPUT_ROOT, "anonymise_dicom_tags_results.parquet")
)

display(result_df)